> **구분: 수업 핵심**

# 04. 구조화 결과와 안전한 실패 처리 — 60분

**자유 문장 문제 10분 → JSON·schema 10분 → parser TODO 20분 → 실제 구조 응답 15분 → 정리 5분** 입니다. 목표는 VLM을 절대적으로 신뢰하는 것이 아니라 허용된 형식만 다음 프로그램으로 넘기는 것입니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

자연어 응답을 프로그램이 사용하려면 정확한 필드·타입·허용값의 계약이 필요합니다. JSON으로 읽힌다는 사실과 장면에 맞는 판단이라는 사실을 나누고, 잘못되거나 불확실한 응답을 무리하게 허용하지 않는 경계를 만듭니다.

### 전체 과정에서 내 코드가 하는 일

```text
모델 원문
  ↓
허용된 JSON 표현 추출
  ↓
schema·값 검증 TODO
  ↓
정상 결과 또는 unknown
  ↓
사람 기대값과 비교
```

바깥 구조부터 안쪽 값을 검사합니다. 모델의 confidence는 자기 보고 값이며 실제 정답 확률로 간주하지 않습니다. false_allow와 불필요한 보류를 따로 기록합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 모든 응답을 unknown으로 바꾸면 좋은 검증기인가요?

**설명:** 잘못된 허용은 줄일 수 있어도 유효한 판단까지 버리므로 작업을 수행하지 못합니다. 정상 결과를 받아들이는 검사와 잘못된 결과를 거부하는 검사를 모두 통과해야 합니다. 형식 통과 뒤에도 의미 평가는 남습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**2일차 VLM 연결:** 실제 질의에는 공용 Ollama가 필요합니다. 연결되지 않거나 모델이 없으면 로컬 Setup 담당자에게 서버·모델 저장소 확인을 요청합니다. 준비 여부는 **개인 컨테이너 JupyterLab Terminal** 에서 확인합니다.

```bash
cd /workspace
/opt/course/bin/python 05.VLM/scripts/check_environment.py
```

기본 API는 `http://127.0.0.1:11434`, 모델은 `gemma4:e2b-it-qat`이며 `vision` capability가 필요합니다. 주소·모델을 따로 배정받았다면 Python 초기화 셀에서 `os.environ['OLLAMA_HOST']`·`os.environ['VLM_MODEL']`을 설정합니다. 다른 Terminal에서 설정한 환경변수는 이미 열린 커널에 자동 적용되지 않습니다. 입력은 `Data/images/05.VLM/{road.png,dog.jpg}`, 선택 영상은 `Data/videos/05.VLM/ETS.mp4`, 출력은 `Data/outputs/05.VLM`입니다. 파일·모델이 없으면 배포 자산 경로를 확인합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[02_image_questions.ipynb의 1·3절](02_image_questions.ipynb)에 질문 계약과 실제 이미지 요청, [03_camera_description.ipynb의 2·3절](03_camera_description.ipynb)에 최소 간격·최신 프레임 설명이 있습니다. 필요한 내용은 질문에 관찰 기준·불확실하면 `unknown`을 명시하고, 한 번에 이미지 요청 하나를 보내며, 답 문자열을 받은 뒤 다음 단계로 넘기기 전에 검사한다는 것입니다. 이 노트북은 배포 `road.png`를 직접 읽어 JSON 문자열을 검사합니다. 이전 message/prompt/sampling TODO·저장 결과를 실행 입력으로 사용하지 않습니다. 마지막 단위 테스트는 제공 공통 코드와 참고 정답의 검증이며 본인의 parser 완료는 `check_results('exercise')`로 확인합니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–6절의 수업 핵심 경로** → [2일차 슬라이드 76쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s071) · [배포 PDF 76쪽](../슬라이드/2일차.pdf#page=76)에서 85쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 같은 이미지의 요청·원문·검증 결과·지연을 저장합니다.
- **단원 마지막 연결 질문:** 공통 사례의 기대 label을 응답을 보기 전에 정합니다. 형식 통과와 업무 판단 정답을 따로 기록하고, 틀린 허용·불필요한 거부·판단 보류를 구분합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

05의 code fence 허용은 표시용 장면 응답의 계약입니다. 09의 행동 입력은 JSON object 전체만 허용하므로 code fence를 거부합니다. 05 parser 결과를 09 명령으로 직접 재사용하지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심**

## 완료 조건

- JSON 문법 검사와 의미 계약 검사를 구분합니다.
- `parse_scene_result`가 정상 입력과 경계값 0·1을 허용하고 잘못된 key·label·숫자·근거를 거부합니다.
- 오류일 때 `unknown`, confidence 0으로 바꾸는 fallback을 확인합니다.
- 실제 구조화 결과와 입력·시간을 Data에 저장합니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys, time
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '05.VLM/common').is_dir())
COURSE = ROOT / '05.VLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import IMAGES, VIDEOS, OUTPUTS, LOGS, ensure_output_dirs
from common.assets import compact_image_info, encode_image
from common.client import OllamaVLMClient, timing_summary
from common.loading import load_op
ensure_output_dirs()
assert (IMAGES / 'road.png').is_file() and (IMAGES / 'dog.jpg').is_file(), '먼저 강사에게 Data/images/05.VLM 배포를 확인하세요.'
print('Python:', sys.executable)
print('input:', IMAGES)
print('output:', OUTPUTS)
from common.structured import ASSESSMENT_SCHEMA, parse_assessment, safe_assessment


> **구분: 수업 핵심**

## 1. 자연어를 곧바로 제어 조건으로 쓰면 안 됩니다

`조심해야 할 것 같습니다`, `주의`, `CAUTION!`은 사람에게 비슷해 보여도 문자열 비교에서는 다릅니다. 반대로 JSON 문법이 맞아도 `label='fly'`, `confidence=27`처럼 시스템 계약에 없는 값일 수 있습니다.

이 단원은 세 field만 허용합니다.

| key | 자료형 | 계약 |
|---|---|---|
| `label` | string | `clear`, `caution`, `unknown` 중 하나 |
| `confidence` | number | bool이 아닌 유한한 0–1 값 |
| `evidence` | string list | 1–3개의 이미지 근거, 각 1–120자 |

`confidence=0`과 `confidence=1`은 정확한 경계이므로 허용합니다. 그러나 이 숫자는 모델의 자기 보고이며 실제 확률로 단정하지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
examples = [
    '{"label":"caution","confidence":0.8,"evidence":["차량이 보임"]}',
    '{"label":"go","confidence":0.8,"evidence":["차량이 보임"]}',
    '{"label":"clear","confidence":true,"evidence":["도로"]}',
    'JSON이 아닌 설명 문장',
]
for text in examples:
    result, error = safe_assessment(text)
    print('\ninput:', text)
    print('usable:', result.to_dict(), 'error:', error)


> **구분: 수업 핵심**

## 2. TODO — 구조와 의미를 모두 검사하기

<!-- procedure:lesson-20-da4a2f:start -->
![형식과 의미를 차례로 검사하기](../교재그림/절차/lesson-20.svg)

5번과 6번은 통과/실패 분기입니다. JSON으로 읽혔다는 사실만으로 장면 판단이 맞는 것은 아닙니다.
<!-- procedure:lesson-20-da4a2f:end -->


[exercises/results.py](exercises/results.py)의 `parse_scene_result(text, allowed_labels)`를 완성하세요. 표의 계약을 만족하면 immutable `SceneAssessment`를 반환하고, 틀리면 `ValueError` 또는 `AssessmentError`를 발생시킵니다.

순수 JSON 또는 첫 줄이 ` ``` `이나 ` ```json `이고 마지막 줄이 ` ``` `인 전체 code fence 안의 JSON을 허용합니다(각 줄 바깥 공백은 무시). fence 본문은 비어 있으면 안 되며 다른 언어 이름·네 개 이상의 backtick·여는 줄이나 닫는 줄 뒤 설명은 거부합니다. 그 밖의 앞뒤 설명, JSON list, 빠진/추가 key, bool confidence, NaN·범위 밖 숫자, 빈 evidence는 거부합니다. `allowed_labels` 입력 원본을 변경하지 않습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `text='{"label":"safe","confidence":0.8,"evidence":["통로가 비어 있음"]}'`, allowed_labels=`['safe','unsafe','unknown']`이면 `SceneAssessment('safe',0.8,('통로가 비어 있음',))`을 반환합니다. evidence 출력은 tuple입니다.
**작성 흐름 힌트:** label 모음 검사 → 제공 `strip_json_fence`로 허용된 fence 정리 → json.loads → 정확히 세 key인지 검사 → label 소속, confidence의 bool 제외·유한성·0–1 검사 → evidence 1–3개, 각 strip 결과 1–120자 검사 → 불변 결과 객체 만들기.
**설명할 지점:** “05에서는 fence를 허용하지만 09의 손 행동 parser는 거부합니다. 두 parser의 계약을 섞지 마세요. confidence는 여기서는 검증할 숫자이지 신뢰할 수 있는 정확도 보장이 아닙니다.”
**필요한 도구·통과 기준:** starter가 `json`, `math`, `AssessmentError`, `SceneAssessment`, `strip_json_fence`를 제공합니다. `allowed_labels`는 문자열 하나가 아닌 비어 있지 않은 label 모음이며 각 label도 비어 있지 않은 문자열이어야 합니다. label을 하드코딩하거나 대소문자를 자동 수정하지 않고 전달받은 허용 목록과 그대로 비교합니다. `strip_json_fence`는 순수 JSON 또는 전체를 감싼 fence만 풀며 JSON 앞뒤 설명은 여전히 파싱 실패입니다. 성공은 정확히 세 필드의 `SceneAssessment`, 실패는 예외입니다. TODO 안에서 모든 실패를 unknown으로 숨기지 않습니다. unknown fallback·실패 기록은 아래 호출 측 역할입니다. `check_results('exercise')`의 정상·경계·오류 검증을 통과하세요.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
from tests.test_course import check_results
check_results('exercise')
print('structured result TODO PASS')


In [ ]:
# 구분: 수업 핵심 셀
# 참고 정답은 강사가 안내하는 시점에 파일 편집기에서 확인합니다.
solution_path = COURSE / 'solutions/results.py'
print('참고 정답 경로:', solution_path)
# 수강생 파일을 자동으로 덮어쓰지 않습니다.


### 정답 해설 — 입력부터 결과까지

**바깥에서 안으로 검증:** JSON 문법→object 여부→정확한 key 집합→허용 label→confidence 숫자·유한성·0–1 범위→evidence 목록과 문자열 길이 순서로 검사합니다. 성공하면 SceneAssessment로 변환하고 잘못된 출력은 AssessmentError로 알려 호출 측의 보류 처리를 가능하게 합니다.

confidence=True는 숫자처럼 보이지만 거부합니다. evidence는 1–3개이고 각 항목은 공백을 정리한 뒤 1–120자여야 합니다. 05는 표시용 JSON code fence를 허용하지만 09의 행동 parser는 허용하지 않습니다.

**해석 예:** road.png에서 `clear/0.95`가 형식 검사를 통과해도 근거에 차량이 있고 과제 규칙이 caution이면 의미 오답입니다. parser는 실제 사진을 보지 않습니다.

**확인 질문:** 형식 오류, 의미 오답, unknown 보류를 왜 각각 기록해야 하나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/results.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""04_structured_results.ipynb 참고 정답."""
import json
import math
from common.structured import AssessmentError, SceneAssessment, strip_json_fence


def parse_scene_result(text, allowed_labels):
    if isinstance(allowed_labels, (str, bytes)):
        raise ValueError("allowed_labels는 문자열 하나가 아닌 label 모음이어야 합니다.")
    labels = tuple(allowed_labels)
    if not labels or any(not isinstance(label, str) or not label for label in labels):
        raise ValueError("allowed_labels에는 비어 있지 않은 문자열이 필요합니다.")
    # 이 단원에서는 순수 JSON 또는 허용된 Markdown fence만 해제해 읽습니다.
    try:
        value = json.loads(strip_json_fence(text))
    except json.JSONDecodeError as exc:
        raise AssessmentError(f"JSON 문법 오류: {exc.msg}") from exc
    if not isinstance(value, dict):
        raise AssessmentError("최상위 결과는 JSON object여야 합니다.")
    if set(value) != {"label", "confidence", "evidence"}:
        raise AssessmentError("key는 정확히 label, confidence, evidence여야 합니다.")
    if value["label"] not in labels:
        raise AssessmentError(f"허용되지 않은 label입니다: {value['label']!r}")
    # Python의 bool과 NaN도 숫자로 섞일 수 있으므로 별도로 거부합니다.
    confidence = value["confidence"]
    if isinstance(confidence, bool) or not isinstance(confidence, (int, float)):
        raise AssessmentError("confidence는 bool이 아닌 숫자여야 합니다.")
    confidence = float(confidence)
    if not math.isfinite(confidence) or not 0 <= confidence <= 1:
        raise AssessmentError("confidence는 0과 1 사이의 유한한 숫자여야 합니다.")
    evidence = value["evidence"]
    if not isinstance(evidence, list) or not 1 <= len(evidence) <= 3:
        raise AssessmentError("evidence는 1~3개의 문자열 list여야 합니다.")
    if any(not isinstance(item, str) or not item.strip() or len(item.strip()) > 120 for item in evidence):
        raise AssessmentError("각 evidence는 1~120자의 문자열이어야 합니다.")
    # evidence는 tuple로 고정해 검증 뒤 결과 내용이 바뀌지 않게 합니다.
    return SceneAssessment(value["label"], confidence, tuple(item.strip() for item in evidence))
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
op = load_op('results')
sample = json.dumps({'label': 'caution', 'confidence': 0.72, 'evidence': ['도로 위 차량 여러 대']}, ensure_ascii=False)
assessment = op.parse_scene_result(sample, ['clear', 'caution', 'unknown'])
print(assessment)
print(assessment.to_dict())


> **구분: 수업 핵심**

## 3. JSON schema는 출력 요청 계약입니다

Ollama 요청의 `format`에 schema를 넣으면 모델이 해당 구조를 생성하도록 돕습니다. 그러나 네트워크 중단, 버전 차이, 예상 밖 문자열을 고려해 client parser는 여전히 필요합니다. **모델에게 요구하는 것** 과 **프로그램이 신뢰하기 전에 확인하는 것** 은 두 단계입니다.


In [ ]:
# 구분: 수업 핵심 셀
print(json.dumps(ASSESSMENT_SCHEMA, ensure_ascii=False, indent=2))


> **구분: 수업 핵심**

## 4. 실제 구조화 이미지 판단과 저장

`road.png`에서 차량이나 사람이 있어 주의가 필요하면 `caution`, 명백히 비어 있으면 `clear`, 판단할 수 없으면 `unknown`을 요청합니다. 이 분류는 교육용이며 실제 차량 제어 규칙이 아닙니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.recording import record_structured_query
client = OllamaVLMClient(timeout=300)
prompt = '도로 장면의 차량·사람이 있으면 caution, 명백히 비어 있으면 clear, 판단할 수 없으면 unknown과 눈에 보이는 근거를 답하세요.'
record, destination = record_structured_query(
    client, IMAGES / 'road.png', prompt, op.parse_scene_result, OUTPUTS)
print(json.dumps(record, ensure_ascii=False, indent=2))
print('saved:', destination)
# status=ok인 unknown은 모델 판단, request_error/format_error의 unknown은 오류 fallback입니다.


> **구분: 수업 핵심**

## 5. 실패는 값과 원인을 함께 남깁니다

fallback을 `unknown`으로 만드는 동시에 오류 문자열을 로그에 남겨야 “실제로 unknown 장면”과 “응답을 검증하지 못함”을 구분할 수 있습니다. 이후 행동 코드는 `unknown`을 정지로 연결하고, 운영자는 오류 원인으로 API·모델·schema를 점검합니다.

### 실제 실행 사례 — `status=ok`인데 판단은 틀릴 수 있습니다

2026-09-08 배포 전 `road.png` 질의는 `status=ok`, `label=clear`, `confidence=0.95`였지만 evidence에는 “도로 위에는 여러 대의 차량이 주행”한다고 썼습니다. 차량·사람이 있으면 `caution`이라는 요청 기준과 `clear`가 모순됩니다. **형식 검사는 통과했지만 장면 판단은 오답** 으로 기록합니다. 이 사례는 당시 관측 결과이며 매 실행에서 같은 문장이 나오는 것은 아닙니다.

`status=ok`는 요청·응답 파싱과 key·자료형·허용값 검증을 통과했다는 뜻입니다. parser는 사진을 다시 보거나 evidence의 사실성·label과의 일관성을 검사하지 않습니다. `confidence=0.95`도 모델의 자기 보고 숫자이며 “95% 확률로 정답”이라는 뜻이 아닙니다.

응답을 보기 전에 사진과 분류 규칙으로 기대 label을 정하고, 저장한 원문·label·evidence를 나란히 비교하세요. 이 사례는 `expected_label=caution`, `model_label=clear`, 형식 통과, 의미 오답이며 평가 워크북에서는 `false_allow`입니다. 높은 confidence를 이유로 오답을 통과시키거나 원문을 정답으로 덮어쓰지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
bad_responses = ['timeout 대신 기록된 빈 응답', '{"label":"run","confidence":0.9,"evidence":["x"]}']
for raw in bad_responses:
    fallback, error = safe_assessment(raw)
    assert fallback.label == 'unknown' and fallback.confidence == 0
    print({'assessment': fallback.to_dict(), 'validation_error': error})


> **구분: 수업 핵심**

## 6. 전체 단위 테스트

이 테스트는 실제 모델의 문장을 고정 정답으로 비교하지 않습니다. 이미지·message·prompt·시간 선택·JSON parser의 결정 가능한 계약을 빠르게 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
completed = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', str(COURSE / 'tests')],
    text=True, capture_output=True,
)
print(completed.stdout)
print(completed.stderr)
assert completed.returncode == 0, '단위 테스트 오류를 위에서 확인하세요.'




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 구조 검증·실제 요청·실패 기록·형식/의미 평가

[2일차 슬라이드 76쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s071) · [배포 PDF 76쪽](../슬라이드/2일차.pdf#page=76)로 돌아가 **76–85쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습**

## 자습: 재시도 정책 질문

서버 timeout 때 같은 오래된 frame을 즉시 10번 재시도하면 어떤 문제가 생길까요? 실시간 시스템에서는 제한된 횟수, 지수 backoff, 다음 최신 frame으로 이동 중 하나를 목적에 맞게 정합니다. 행동 명령은 별도의 유효 시간도 가져야 합니다.


> **구분: 수업 핵심**

## 05.VLM 정리와 09.Pseudo-VLA 연결

완성된 흐름은 `image → 검증된 bytes → VLM request → JSON text → SceneAssessment → unknown fallback → Data log`입니다. 09.Pseudo-VLA에서는 이 판단 결과를 허용 동작 목록과 다시 대조합니다. `confidence` 하나나 자유 문장을 곧바로 속도 명령으로 바꾸지 않습니다.
